## HRDPA sample extraction
This notebook downloads a single HRDPA 6-hour precipitation analysis, clips it to the Nicomekl 08MH155 watershed polygon (`g`), and prepares a joined daily table with Water Survey of Canada discharge data.


In [ ]:
# Imports, configuration, and lightweight dependency checks
from pathlib import Path
import numpy as np
import pandas as pd
import requests
import xarray as xr
import matplotlib.pyplot as plt

try:
    import cfgrib  # noqa: F401
except ImportError as exc:
    raise ImportError(
        "cfgrib is required to read HRDPA GRIB2 files. Install cfgrib and eccodes in the msc_open_data environment."
    ) from exc

try:
    import eccodes  # noqa: F401
except ImportError as exc:
    raise ImportError(
        "The eccodes Python bindings are required for cfgrib. Install eccodes in the msc_open_data environment."
    ) from exc

try:
    import rioxarray  # noqa: F401
except ImportError as exc:
    raise ImportError(
        "rioxarray is required to attach and use CRS information for clipping the HRDPA grid."
    ) from exc

plt.style.use("ggplot")
raw_grib = Path("../data/raw/hrdpa/HRDPA_20240501_00_accum6h.grib2")
feat_out = Path("../data/processed/features/hrdpa_sample.parquet")
joined_out = Path("../data/model_ready/train_sample.parquet")
for folder in {raw_grib.parent, feat_out.parent, joined_out.parent}:
    folder.mkdir(parents=True, exist_ok=True)


In [ ]:
# Download the HRDPA 6-hour accumulation file if not already cached
hrdpa_url = (
    "https://dd.weather.gc.ca/analysis/precip/hrdpa/grib2/06/000/"
    "CMC_HRDPA_2.5f_Pcpn6hr_SFC_0_latlon0p025x0p025_2024050100_P000.grib2"
)
if not raw_grib.exists():
    print(f"Downloading HRDPA file from {hrdpa_url} ...")
    response = requests.get(hrdpa_url, stream=True, timeout=120)
    response.raise_for_status()
    with raw_grib.open("wb") as f:
        for chunk in response.iter_content(chunk_size=1048576):
            if chunk:
                f.write(chunk)
    print(f"Saved to {raw_grib}")
else:
    print(f"Using cached file at {raw_grib}")


In [ ]:
# Open, clip, and aggregate the 6-hour precipitation field
# Assumes `g` is a GeoDataFrame for Nicomekl 08MH155 in EPSG:4326.
ds = xr.open_dataset(raw_grib, engine="cfgrib")
print(ds)

precip_candidates = [name for name in ds.data_vars if "tp" in name.lower() or "precip" in name.lower()]
precip_var = precip_candidates[0] if precip_candidates else list(ds.data_vars)[0]
print(f"Selected precip variable: {precip_var}")

p_field = ds[precip_var]
dim_map = {}
for dim in p_field.dims:
    lower = dim.lower()
    if lower.startswith("lon") or lower in {"x"}:
        dim_map["x"] = dim
    elif lower.startswith("lat") or lower in {"y"}:
        dim_map["y"] = dim
if {"x", "y"} - dim_map.keys():
    raise ValueError(f"Unable to infer spatial dimensions from dims: {p_field.dims}")

units = (p_field.attrs.get("units") or "").lower()
if "kg" in units and "m" in units:
    # 1 kg m-2 of liquid water equals 1 mm.
    precip_to_mm = 1.0
    unit_note = "Converted kg m-2 to mm (1:1)."
elif units in {"m", "metre", "meter"} or units.endswith(" m"):
    precip_to_mm = 1000.0
    unit_note = "Converted metres to mm by multiplying by 1000."
else:
    precip_to_mm = 1.0
    unit_note = "Assuming source data already in mm."
print(unit_note)

p_field_mm = (p_field * precip_to_mm).rio.set_spatial_dims(
    x_dim=dim_map["x"], y_dim=dim_map["y"], inplace=False
).rio.write_crs("EPSG:4326", inplace=False)

# Clip to the watershed geometry and compute the spatial mean.
ds_clip = p_field_mm.rio.clip(g.geometry, g.crs, drop=True).to_dataset(name="P_mm")
spatial_dims = [dim_map["y"], dim_map["x"]]
p6h = ds_clip["P_mm"].mean(dim=spatial_dims)

if {"time", "step"}.issubset(p6h.coords):
    valid_time = (p6h["time"] + p6h["step"]).rename("valid_time")
    p6h = p6h.assign_coords(valid_time=valid_time)
    if "time" in p6h.dims:
        p6h = p6h.swap_dims({"time": "valid_time"})
    p6h = p6h.drop_vars("step")
elif "time" in p6h.coords and "time" in p6h.dims:
    p6h = p6h.rename({"time": "valid_time"})

if "valid_time" in p6h.coords:
    p6h = p6h.rename({"valid_time": "time"})

p6h = p6h.sortby("time").rename("P_mm_6h")
print(p6h.to_series().head())

p_daily = (
    p6h.to_series()
    .to_frame(name="P_mm")
    .rename_axis("time")
    .sort_index()
)
p_daily.index = pd.to_datetime(p_daily.index, utc=True)
p_daily = p_daily.resample("1D").sum(min_count=1)
print(p_daily.head())


In [ ]:
# Fetch 08MH155 discharge, align windows, and compute daily means
flow_url = "https://wateroffice.ec.gc.ca/services/real_time_data/csv/08MH155/Q"
q_df = pd.read_csv(flow_url, comment="#")
column_map = {}
for candidate in q_df.columns:
    lower = candidate.lower().replace(" ", "")
    if lower in {"datetime", "date_time", "date"}:
        column_map["ts"] = candidate
    elif lower in {"dateutc", "datetimeutc"}:
        column_map["ts"] = candidate
    elif lower in {"value", "q", "discharge"}:
        column_map["value"] = candidate
if "ts" not in column_map or "value" not in column_map:
    raise ValueError(f"Unexpected Water Survey CSV columns: {list(q_df.columns)}")

q_hourly = (
    q_df.rename(columns={column_map["ts"]: "ts", column_map["value"]: "Q_m3s"})
    .assign(ts=lambda d: pd.to_datetime(d["ts"], utc=True))
    .dropna(subset=["ts", "Q_m3s"])
    .set_index("ts")
    .sort_index()
)

window_start = p_daily.index.min() - pd.Timedelta(days=3)
window_end = p_daily.index.max() + pd.Timedelta(days=3)
q_hourly = q_hourly.loc[window_start:window_end]
q_daily = q_hourly.resample("1D").mean()
print(q_daily.head())


In [ ]:
# Join, log summary statistics, persist parquet artifacts, and plot
joined = p_daily.join(q_daily, how="left")

summary = {
    "p6h": {
        "shape": tuple(int(s) for s in np.atleast_1d(p6h.shape)),
        "min": float(np.nanmin(p6h.values)),
        "max": float(np.nanmax(p6h.values)),
    },
    "p_daily": {
        "shape": tuple(p_daily.shape),
        "min": float(np.nanmin(p_daily["P_mm"].values)),
        "max": float(np.nanmax(p_daily["P_mm"].values)),
    },
    "q_hourly": {
        "shape": tuple(q_hourly.shape),
        "min": float(np.nanmin(q_hourly["Q_m3s"].values)) if not q_hourly.empty else float("nan"),
        "max": float(np.nanmax(q_hourly["Q_m3s"].values)) if not q_hourly.empty else float("nan"),
    },
    "q_daily": {
        "shape": tuple(q_daily.shape),
        "min": float(np.nanmin(q_daily["Q_m3s"].values)) if not q_daily.empty else float("nan"),
        "max": float(np.nanmax(q_daily["Q_m3s"].values)) if not q_daily.empty else float("nan"),
    },
    "joined": {
        "shape": tuple(joined.shape),
        "min_P": float(np.nanmin(joined["P_mm"].values)) if not joined.empty else float("nan"),
        "max_P": float(np.nanmax(joined["P_mm"].values)) if not joined.empty else float("nan"),
        "min_Q": float(np.nanmin(joined["Q_m3s"].values)) if not joined.empty else float("nan"),
        "max_Q": float(np.nanmax(joined["Q_m3s"].values)) if not joined.empty else float("nan"),
    },
}
for name, stats in summary.items():
    print(name, stats)

p_daily.to_parquet(feat_out)
joined.to_parquet(joined_out)
print(f"Wrote {feat_out} ({len(p_daily)} rows)")
print(f"Wrote {joined_out} ({len(joined)} rows)")

fig, ax = plt.subplots(figsize=(8, 4))
joined["Q_m3s"].plot(ax=ax, label="Daily Q (m³/s)", color="tab:blue", marker="o")
ax.set_ylabel("Discharge (m³/s)")
ax2 = ax.twinx()
joined["P_mm"].plot(ax=ax2, label="Daily P (mm)", color="tab:orange", marker="s")
ax2.set_ylabel("Precipitation (mm)")
ax.set_title("Daily discharge and HRDPA precipitation")
ax.legend(loc="upper left")
ax2.legend(loc="upper right")
plt.tight_layout()
plt.show()
